# Ambi — detector local RF-DETR Nano

Entrena pesos propios con RF-DETR Nano (Apache 2.0) sobre el dataset público de bobwatson (CC BY 4.0). No reutiliza ni redistribuye el checkpoint YOLOv11 alojado en Roboflow.

Dataset: `bobwatson/clash-royale-cylln-88wu8-yeqpq`, versión 1, 145 clases. Ejecutar en Google Colab con GPU T4 o superior.

In [ ]:
!pip -q install "rfdetr[train,onnx]" roboflow supervision

In [ ]:
import os, json, getpass, shutil, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Activá GPU en Entorno de ejecución > Cambiar tipo de entorno.'
print('GPU:', torch.cuda.get_device_name(0))
api_key = getpass.getpass('Roboflow API key (entrada oculta): ')
from roboflow import Roboflow
rf = Roboflow(api_key=api_key)
project = rf.workspace('bobwatson').project('clash-royale-cylln-88wu8-yeqpq')
dataset = project.version(1).download('coco')
DATA = Path(dataset.location)
for split in ('train','valid','test'):
    assert (DATA / split / '_annotations.coco.json').exists(), f'Falta split {split}; no se acepta sustituir test por valid.'
print('Dataset descargado:', DATA)


In [ ]:
from rfdetr import RFDETRNano
OUT = Path('/content/ambi-rfdetr-output')
model = RFDETRNano()  # checkpoint base COCO de RF-DETR, no pesos YOLO del proyecto
model.train(dataset_dir=str(DATA), epochs=80, batch_size=4, grad_accum_steps=4, lr=1e-4, output_dir=str(OUT), eval_interval=1)


In [ ]:
# Evaluación en el split test retenido, separado de train y valid.
metrics = model.evaluate(dataset_dir=str(DATA), split='test')
metrics_out = {k: float(v) for k,v in metrics.items() if isinstance(v,(int,float))}
(OUT / 'test_metrics.json').write_text(json.dumps(metrics_out, indent=2))
print(json.dumps(metrics_out, indent=2))


In [ ]:
# Exportar ONNX y empaquetar con el orden de clases del COCO exportado.
EXPORT = OUT / 'export'
model.export(format='onnx', output_dir=str(EXPORT))
onnx_files = list(EXPORT.glob('*.onnx')) or list(OUT.rglob('*.onnx'))
assert len(onnx_files) == 1, f'ONNX esperado: 1 archivo; encontrados: {onnx_files}'
ann = json.loads((DATA/'train'/'_annotations.coco.json').read_text())
cats = sorted(ann['categories'], key=lambda c:c['id'])
classes = [c['name'] for c in cats]
assert len(classes) == 145, f'Cambió el número de clases: {len(classes)}'
PKG = Path('/content/ambi-local-detector'); PKG.mkdir(exist_ok=True)
shutil.copy2(onnx_files[0], PKG/'arena_detector.onnx')
(PKG/'arena_classes.json').write_text(json.dumps(classes, indent=2))
(PKG/'DATASET_ATTRIBUTION.txt').write_text('Dataset: Clash royale by bobwatson, Roboflow Universe, version 1.\nURL: https://universe.roboflow.com/bobwatson/clash-royale-cylln-88wu8-yeqpq\nDataset license: CC BY 4.0. Attribute the author and link when redistributing derivatives.\nDetector architecture/package: RF-DETR Nano, Apache 2.0.\n')
with zipfile.ZipFile('/content/ambi-local-detector.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in PKG.iterdir(): z.write(f, f.name)
print('LISTO:', '/content/ambi-local-detector.zip', 'tamaño:', Path('/content/ambi-local-detector.zip').stat().st_size)


## Antes de integrarlo en la APK

Descargá `ambi-local-detector.zip` desde Archivos de Colab y adjuntalo en el chat. El siguiente control comprueba el contrato real de tensores ONNX, las métricas de test y el tamaño del modelo; después hay que medir latencia y RAM en el Moto e32. Terminar el entrenamiento no equivale a validar el detector en el teléfono.